# 04: Random Sampling, Sorting & Searching (Exercises 46–60)

Explore modern PRNG generation, Cauchy matrices, pairwise distance geometry, nested records, and argsort permutations.

---


In [ ]:
import numpy as np
print(f"NumPy version: {np.__version__}")

### Exercise 46: Create a structured array with x and y coordinates covering [0,1]x[0,1]
**Difficulty:** `★★☆`  
**Tags:** `Structured Arrays, Meshgrid`

#### 💡 Intuition & Concept
Structured dtypes can hold composite schemas like `[('x', float), ('y', float)]`. `np.meshgrid` creates 2D evaluation grids.

#### ⚠️ Key Takeaway & Gotchas
Assigning meshgrid outputs into the structured fields maps 2D coordinates into a single unified array.


In [ ]:
Z = np.zeros((5, 5), [('x', float), ('y', float)])
Z['x'], Z['y'] = np.meshgrid(np.linspace(0, 1, 5), np.linspace(0, 1, 5))
print(Z['x'][:2, :2])
print(Z[0, 0])

### Exercise 47: Given two arrays, X and Y, construct Cauchy matrix C (Cij = 1/(xi - yj))
**Difficulty:** `★★☆`  
**Tags:** `Broadcasting, Outer Operations`

#### 💡 Intuition & Concept
The outer difference `np.subtract.outer(X, Y)` computes all pairwise differences $x_i - y_j$ in a vectorized 2D matrix, allowing direct inversion.

#### ⚠️ Key Takeaway & Gotchas
If any $x_i == y_j$, a division by zero occurs.


In [ ]:
X = np.arange(5)
Y = X + 0.5
C = 1.0 / np.subtract.outer(X, Y)
print("Cauchy Matrix shape:", C.shape)
print(np.round(C, 2))

### Exercise 48: Print minimum and maximum representable value for each numpy scalar type
**Difficulty:** `★★☆`  
**Tags:** `Types, iinfo, finfo`

#### 💡 Intuition & Concept
`np.iinfo` inspects limits for integer dtypes, while `np.finfo` inspects machine limits (min, max, epsilon) for floating point types.

#### ⚠️ Key Takeaway & Gotchas
Floating point `min` is the smallest *positive* normal number, while for integers `min` is the most negative number.


In [ ]:
print("--- Integer Limits ---")
for dtype in [np.int8, np.int16, np.int32, np.int64]:
    info = np.iinfo(dtype)
    print(f"{dtype.__name__:8s} -> Min: {info.min:20d}, Max: {info.max:20d}")

print("\n--- Float Limits ---")
for dtype in [np.float32, np.float64]:
    f_info = np.finfo(dtype)
    print(f"{dtype.__name__:8s} -> Min: {f_info.min:.2e}, Max: {f_info.max:.2e}, Eps: {f_info.eps:.2e}")

### Exercise 49: How to print all values of an array without truncation?
**Difficulty:** `★☆☆`  
**Tags:** `Display Options, Printing`

#### 💡 Intuition & Concept
NumPy truncates large arrays when printing to keep console output readable. `np.set_printoptions(threshold=np.inf)` disables truncation.

#### ⚠️ Key Takeaway & Gotchas
Resetting it back with `np.set_printoptions(threshold=1000)` prevents runaway outputs for huge data.


In [ ]:
# Temporarily print without truncation
with np.printoptions(threshold=np.inf):
    Z = np.arange(50).reshape(5, 10)
    print(Z)

### Exercise 50: How to find closest value (to a given scalar) in a vector?
**Difficulty:** `★☆☆`  
**Tags:** `Searching, Argmin`

#### 💡 Intuition & Concept
Compute the absolute distance array `np.abs(Z - target)`, and find the index with minimum distance using `argmin()`.

#### ⚠️ Key Takeaway & Gotchas
Works on both 1D and flattened arrays using `.flat[idx]`.


In [ ]:
rng = np.random.default_rng(seed=42)
Z = rng.uniform(0, 100, 10)
target = 50.0
idx = np.abs(Z - target).argmin()
print(f"Target: {target}")
print(f"Closest value in array: {Z[idx]:.4f} at index {idx}")

### Exercise 51: Create a structured array representing position (x,y) and color (r,g,b)
**Difficulty:** `★★☆`  
**Tags:** `Structured Arrays, Nested Schemas`

#### 💡 Intuition & Concept
NumPy structured arrays support nested compound types, mimicking nested C structures.

#### ⚠️ Key Takeaway & Gotchas
Fields can be accessed independently as `arr['position']['x']` or `arr['color']['r']`.


In [ ]:
nested_dtype = np.dtype([
    ('position', [('x', float), ('y', float)]),
    ('color',    [('r', float), ('g', float), ('b', float)])
])
Z = np.zeros(3, dtype=nested_dtype)
Z[0] = ((1.5, 2.5), (255, 0, 0))
print(Z)
print("Position:", Z['position'][0])

### Exercise 52: Consider random vector shape (100,2) coordinates, find point by point distances
**Difficulty:** `★★☆`  
**Tags:** `Broadcasting, Pairwise Distance`

#### 💡 Intuition & Concept
Using broadcasting with `newaxis`: expand dimension to `(100, 1)` and `(1, 100)` to evaluate differences across all pairs without explicit Python loops.

#### ⚠️ Key Takeaway & Gotchas
The distance matrix has shape `(100, 100)` and is symmetric with zeros on the diagonal.


In [ ]:
rng = np.random.default_rng(seed=42)
Z = rng.random((10, 2))
X, Y = np.atleast_2d(Z[:, 0]), np.atleast_2d(Z[:, 1])
D = np.hypot(X - X.T, Y - Y.T)
print("Distance matrix shape:", D.shape)
print("First 3x3 distances:\n", np.round(D[:3, :3], 3))

### Exercise 53: How to convert float (32 bits) array into integer (32 bits) in place?
**Difficulty:** `★★☆`  
**Tags:** `Views, Memory, In-place`

#### 💡 Intuition & Concept
Using `view(np.int32)` reinterprets the raw memory buffer. To convert values rather than bitwise reinterpreting, use `view` and assign converted values.

#### ⚠️ Key Takeaway & Gotchas
A bare `.astype()` creates a brand new copy and fails the in-place requirement.


In [ ]:
Z = np.arange(10, dtype=np.float32)
print("Original dtype:", Z.dtype)
# In-place view & assignment
Y = Z.view(np.int32)
Y[:] = Z
print("Converted dtype:", Y.dtype)
print("Values:", Y)

### Exercise 54: How to read text data using genfromtxt?
**Difficulty:** `★☆☆`  
**Tags:** `I/O, Parsing`

#### 💡 Intuition & Concept
`np.genfromtxt` parses tabular data from files or string buffers (`io.StringIO`), automatically handling missing values.

#### ⚠️ Key Takeaway & Gotchas
For high-performance file reading in modern pipelines, consider `polars` or `pandas`, but `np.genfromtxt` is built right into NumPy.


In [ ]:
from io import StringIO

data = StringIO('''1, 2, 3, 4, 5
6,  ,  , 7, 8
 ,  , 9, 10, 11''')

Z = np.genfromtxt(data, delimiter=",", filling_values=0)
print(Z)

### Exercise 55: What is the equivalent of enumerate for numpy arrays?
**Difficulty:** `★☆☆`  
**Tags:** `Iteration, ndenumerate`

#### 💡 Intuition & Concept
`np.ndenumerate(arr)` yields pairs of multi-dimensional indices and values. `np.ndindex(shape)` yields just the coordinates.

#### ⚠️ Key Takeaway & Gotchas
Direct vectorized operations are always much faster than iterating with `ndenumerate` in Python.


In [ ]:
Z = np.arange(4).reshape(2, 2)
for idx, val in np.ndenumerate(Z):
    print(f"Index {idx} -> Value {val}")

### Exercise 56: Generate a generic 2D Gaussian-like array
**Difficulty:** `★★☆`  
**Tags:** `Math, Kernel`

#### 💡 Intuition & Concept
A 2D isotropic Gaussian kernel is defined as $G(x, y) = \exp\left(-\frac{(x-\mu)^2 + (y-\mu)^2}{2\sigma^2}\right)$.

#### ⚠️ Key Takeaway & Gotchas
Commonly used in image processing for Gaussian blur and spatial filtering.


In [ ]:
sigma, mu = 1.0, 0.0
x, y = np.meshgrid(np.linspace(-1, 1, 5), np.linspace(-1, 1, 5))
d = np.hypot(x - mu, y - mu)
G = np.exp(-(d**2 / (2.0 * sigma**2)))
print("2D Gaussian:\n", np.round(G, 3))

### Exercise 57: How to randomly place p elements in a 2D array?
**Difficulty:** `★★☆`  
**Tags:** `Random, np.put`

#### 💡 Intuition & Concept
`np.put(arr, indices, values)` inserts values into an array using flattened 1D indices, perfectly suited for randomly placing tokens or particles.

#### ⚠️ Key Takeaway & Gotchas
Using `choice(..., replace=False)` guarantees that items are placed in unique, non-overlapping coordinates.


In [ ]:
n, p = 6, 4
Z = np.zeros((n, n), dtype=int)
rng = np.random.default_rng(seed=42)
indices = rng.choice(n * n, p, replace=False)
np.put(Z, indices, 1)
print(Z)

### Exercise 58: Subtract the mean of each row of a matrix
**Difficulty:** `★★☆`  
**Tags:** `Broadcasting, Mean Centering`

#### 💡 Intuition & Concept
Use `X.mean(axis=1, keepdims=True)` to preserve the 2D column dimension `(M, 1)`, allowing direct broadcasted subtraction from `(M, N)`.

#### ⚠️ Key Takeaway & Gotchas
Without `keepdims=True`, `X.mean(axis=1)` returns shape `(M,)`, which broadcasts across columns instead of rows unless transposed.


In [ ]:
rng = np.random.default_rng(seed=42)
X = rng.random((3, 4))
Y = X - X.mean(axis=1, keepdims=True)
print("Original rows means:   ", X.mean(axis=1))
print("Centered rows new means:", np.round(Y.mean(axis=1), 10))

### Exercise 59: How to sort an array by the nth column?
**Difficulty:** `★★☆`  
**Tags:** `Argsort, Column Sorting`

#### 💡 Intuition & Concept
`arr[:, col].argsort()` returns the row permutation indices that sort the target column. Applying this index slice rearranges rows.

#### ⚠️ Key Takeaway & Gotchas
To sort stably by multiple columns, use `np.lexsort`.


In [ ]:
rng = np.random.default_rng(seed=123)
Z = rng.integers(0, 10, (4, 3))
print("Original:\n", Z)
# Sort by 2nd column (index 1)
sorted_Z = Z[Z[:, 1].argsort()]
print("Sorted by column 1:\n", sorted_Z)

### Exercise 60: How to tell if a given 2D array has null columns?
**Difficulty:** `★★☆`  
**Tags:** `Boolean Reductions, Any`

#### 💡 Intuition & Concept
`Z.any(axis=0)` returns `True` for columns containing at least one non-zero. Negating this with `~` finds columns that are all zero. Calling `.any()` checks if at least one such column exists.

#### ⚠️ Key Takeaway & Gotchas
Vectorized boolean reductions evaluate all columns simultaneously in C.


In [ ]:
Z = np.array([
    [0, 1, 0],
    [0, 2, 0],
    [0, 3, 0]
])
has_null_col = (~Z.any(axis=0)).any()
print("Array:\n", Z)
print("Has null column?", has_null_col)